# Phase 13: CIFAR-100 TickNet-L Large v2 (14 khối) + CutMix α=1,0

TickNet-L gốc của nhóm có 7 khối. Theo cách TickNet-small trong bài báo
tăng từ 5 lên 10 khối, bản v2 tăng thành 14 khối, phân bổ theo stage
1–2–5–5–1. Bản này giữ nguyên độ rộng của TickNet-L Large Phase 12 và
giữ khối PDP cải tiến của nhóm; chỉ mượn ý tưởng tăng gấp đôi số khối,
không sao chép lịch kênh riêng của TickNet-small gốc.
Đây là thí nghiệm đối chứng với Phase 9 α=1,0: **chỉ đổi kiến trúc** từ
TickNet-L sang TickNet-L Large v2. Công thức train giữ SGD lr=0.175,
weight decay=0.0005, RandAugment N=2/M=7, CutMix α=1,0 p=0.5,
batch 128, seed 42, 45.000 train / 5.000 validation, cosine LR **200 epochs**.
Không dùng EMA. TickNet-L Large v2 có 4.545.900 tham số và 0,7888G FLOPs
cho một ảnh 32×32, dưới giới hạn 6M tham số / 1G FLOPs.
Phase 9 α=1,0: Best Val 77,44%, Test Top-1 77,66%. Chọn checkpoint bằng
validation Top-1; test chỉ đánh giá một lần sau khi train đủ 200 epochs.

Model Large v2 có kiến trúc khác nên phải chạy **mới từ epoch 1**. Không resume
checkpoint Phase 9/10/11/12. Nếu cần chia phiên, dùng recovery ZIP Phase 13
với cùng ZIP mã nguồn, runtime và GPU.

Upload `TickNets_CIFAR100_L_LargeV2_CutMix_200_Source.zip` làm Kaggle Dataset
và **Add Input**. Chọn GPU và bật Internet; notebook tự tải CIFAR-100 chính thức.
Nguồn có thể là ZIP hoặc thư mục Kaggle đã giải nén. Không cần gắn CIFAR-100
riêng. File kết quả là
`phase13_cifar100_l_large_v2_cutmix_a10_200ep_results.zip` trong `/kaggle/working`.


In [ ]:
from pathlib import Path

SOURCE_ZIP = None       # Exact path if several source copies are attached
SOURCE_ROOT = None      # Exact extracted project folder if needed
KAGGLE_INPUT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
EPOCHS_PER_SESSION = None  # Example: 25 epochs per session
RESUME_ROOT = None         # Folder containing the named Phase 13 run


In [ ]:
import hashlib, importlib.util, json, os, sys, zipfile
from pathlib import Path, PurePosixPath

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
SOURCE_PATHS = ['configs/final/cifar100_sgd_lr0175_wd0005_ra_cutmix.json', 'configs/final/cifar100_ticknet_l_large_v2_cutmix_a10_200.json', 'download_cifar.py', 'models/cifar_data.py', 'models/cifar_experiment.py', 'models/cifar_training.py', 'models/cifar_training_large_v2.py', 'models/common.py', 'models/datasets.py', 'models/model_profile.py', 'models/reproducibility.py', 'models/SE_Attention.py', 'models/TickNet.py', 'models/ticknet_l.py', 'models/ticknet_l_large_v2.py', 'scripts/aggregate_grid_search.py', 'scripts/download_cifar.py', 'scripts/run_kaggle_cutmix.py', 'scripts/run_kaggle_cutmix_large_v2_200.py', 'scripts/verify_cifar_run.py', 'train_cifar.py', 'train_cifar_cutmix.py', 'train_cifar_cutmix_large_v2.py', 'train_cifar_randaugment.py']
PROJECT = WORKING_ROOT / 'TickNets_LLargeV2CutMix200'

def project_prefixes(archive):
    names = {info.filename.replace('\\', '/') for info in archive.infolist() if not info.is_dir()}
    prefixes = []
    for name in names:
        if name == 'train_cifar_cutmix_large_v2.py' or name.endswith('/train_cifar_cutmix_large_v2.py'):
            prefix = name[:-len('train_cifar_cutmix_large_v2.py')]
            if all(prefix + relative in names for relative in SOURCE_PATHS):
                prefixes.append(prefix)
    return sorted(set(prefixes))

def find_project_source():
    if SOURCE_ZIP is not None and SOURCE_ROOT is not None:
        raise ValueError('Set only one of SOURCE_ZIP or SOURCE_ROOT')
    if SOURCE_ZIP is not None:
        candidate = Path(SOURCE_ZIP)
        if not candidate.is_file():
            raise FileNotFoundError(f'Source ZIP not found: {candidate}')
        with zipfile.ZipFile(candidate) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError(f'Source ZIP must contain one complete project; found {prefixes}')
        return 'zip', candidate, prefixes[0]
    if SOURCE_ROOT is not None:
        candidate = Path(SOURCE_ROOT)
        if not candidate.is_dir() or not all((candidate / relative).is_file() for relative in SOURCE_PATHS):
            raise FileNotFoundError(f'Source folder is incomplete: {candidate}')
        return 'directory', candidate, ''
    zip_candidates = []
    directory_candidates = []
    visited = set()
    for current, directories, files in os.walk(KAGGLE_INPUT, followlinks=True):
        resolved = Path(current).resolve()
        if resolved in visited:
            directories[:] = []
            continue
        visited.add(resolved)
        if 'train_cifar_cutmix_large_v2.py' in files and all(
                (Path(current) / relative).is_file() for relative in SOURCE_PATHS):
            directory_candidates.append(Path(current))
            directories[:] = []
            continue
        for filename in files:
            if not filename.lower().endswith('.zip'):
                continue
            candidate = Path(current) / filename
            try:
                with zipfile.ZipFile(candidate) as archive:
                    for prefix in project_prefixes(archive):
                        zip_candidates.append((candidate, prefix))
            except zipfile.BadZipFile:
                continue
    if len(zip_candidates) == 1:
        candidate, prefix = zip_candidates[0]
        return 'zip', candidate, prefix
    if not zip_candidates and len(directory_candidates) == 1:
        return 'directory', directory_candidates[0], ''
    attached = sorted(path.name for path in Path(KAGGLE_INPUT).iterdir()) if Path(KAGGLE_INPUT).is_dir() else []
    raise RuntimeError('Could not identify one complete TickNets source in Kaggle Input. '
                       f'ZIP matches: {zip_candidates}; folder matches: {directory_candidates}; '
                       f'Input entries: {attached}. Attach the source dataset, or set '
                       'SOURCE_ZIP / SOURCE_ROOT to its exact path.')

source_type, source_path, prefix = find_project_source()
if source_type == 'zip':
    with zipfile.ZipFile(source_path) as archive:
        members = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            normalized = info.filename.replace('\\', '/')
            if normalized in members:
                raise ValueError('Source ZIP contains duplicate filenames')
            members[normalized] = info.filename
        manifest_name = prefix + 'cutmix_large_v2_200_source_manifest.json'
        manifest = json.loads(archive.read(members[manifest_name])) if manifest_name in members else None
        payloads = {relative: archive.read(members[prefix + relative]) for relative in SOURCE_PATHS}
else:
    manifest_path = source_path / 'cutmix_large_v2_200_source_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else None
    payloads = {relative: (source_path / relative).read_bytes() for relative in SOURCE_PATHS}
if manifest is not None and set(manifest) != set(SOURCE_PATHS):
    raise ValueError('Source manifest does not match the required files')
PROJECT.mkdir(parents=True, exist_ok=True)
for relative, payload in payloads.items():
    parts = PurePosixPath(relative).parts
    if not parts or any(part in ('', '.', '..') for part in parts):
        raise ValueError(f'Unsafe source path: {relative}')
    if manifest is not None and hashlib.sha256(payload).hexdigest() != manifest[relative]:
        raise ValueError(f'Source checksum differs: {relative}')
    target = (PROJECT / Path(*parts)).resolve()
    if not target.is_relative_to(PROJECT.resolve()):
        raise ValueError(f'Source path escapes project: {relative}')
    if target.exists() and target.read_bytes() != payload:
        raise RuntimeError(f'Existing extracted source differs: {target}; use a fresh session')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
missing = [module for module in ('torch', 'torchvision', 'numpy', 'PIL', 'pandas')
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f'Missing Kaggle packages: {missing}')
import torch, torchvision
assert hasattr(torchvision.transforms, 'RandAugment'), 'torchvision RandAugment is unavailable'
assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running.'
print('Project source:', source_type, source_path)
print('TickNet-L Large v2 source:', hashlib.sha256((PROJECT / 'train_cifar_cutmix_large_v2.py').read_bytes()).hexdigest())
print('Runtime:', torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


In [ ]:
import subprocess
subprocess.run([sys.executable, 'download_cifar.py', '--dataset', 'cifar100',
                '--data-root', str(WORKING_ROOT / 'data')], cwd=PROJECT, check=True)
command = [sys.executable, 'scripts/run_kaggle_cutmix_large_v2_200.py']
if EPOCHS_PER_SESSION is not None:
    command += ['--epochs-per-session', str(EPOCHS_PER_SESSION)]
if RESUME_ROOT is not None:
    command += ['--resume-root', str(RESUME_ROOT)]
subprocess.run(command, cwd=PROJECT, check=True)
